# Training LSTM — Penerjemah BISINDO (word-level)

Notebook ini melatih model **LSTM** untuk mengenali 40 kata BISINDO dari
sequence landmark tangan. Dirancang untuk dijalankan di **Google Colab (GPU)**.

**Pipeline data (dari tahap preprocessing):**
`video .mp4 → MediaPipe Holistic (tangan) → (30 frame, 126 fitur) → X.npy / y.npy`

`n_features = 126` = 2 tangan × 21 titik × 3 (x, y, z).

---
## ⚠️ Catatan untuk penguji: risiko overfitting
Dataset kecil (~50 video/kelas × 40 ≈ 2000 sample, **single-signer**) sehingga
LSTM mudah **menghafal** data latih. Indikatornya: akurasi *train* tinggi tapi
akurasi *validation* mentok/turun (gap melebar).

**Mitigasi yang dipakai di notebook ini:**
- Dropout 0.3–0.4 + `recurrent_dropout` pada LSTM
- Unit moderat (128 → 64), tidak berlebihan
- `EarlyStopping` (restore bobot terbaik) + `ReduceLROnPlateau`
- Split **stratified** 70/15/15 agar tiap kelas seimbang

**Jika akurasi validation rendah**, set `AUGMENT = True` untuk augmentasi landmark
(jitter, scaling, translasi, time-shift, mirror + tukar tangan). Augmentasi murah
karena bekerja pada koordinat, bukan piksel.

## 1. Setup & cek GPU

In [ ]:
import os, json, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "TIDAK ADA -> Runtime > Change runtime type > T4 GPU")

def set_seeds(seed=42):
    """Kunci seed agar hasil reproducible (penting untuk laporan)."""
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

## 2. Konfigurasi terpusat
Nilai mirror dari `config.py` proyek (notebook self-contained).

In [ ]:
SEQUENCE_LENGTH  = 30      # jumlah frame per sample
N_FEATURES       = 126     # 2 tangan x 21 titik x 3 (x,y,z)
NUM_CLASSES      = 40
HAND_FEATURE_LEN = 63      # 21 x 3 per tangan (untuk augmentasi mirror/swap)

BATCH_SIZE    = 32
EPOCHS        = 100
LEARNING_RATE = 1e-3
SEED          = 42

AUGMENT = False            # set True jika akurasi validation rendah

# Path input (sesuaikan saat di Colab; mis. hasil mount Google Drive)
X_PATH         = "X.npy"
Y_PATH         = "y.npy"
LABEL_MAP_PATH = "label_map.json"

OUTPUT_DIR = "outputs"     # tempat artefak (model, grafik, report)
os.makedirs(OUTPUT_DIR, exist_ok=True)

set_seeds(SEED)
print("Config siap. AUGMENT =", AUGMENT)

## 3. Muat dataset (X.npy, y.npy)
Pilih salah satu cara taruh data di Colab:
- **Upload manual**: `from google.colab import files; files.upload()`
- **Google Drive**: `from google.colab import drive; drive.mount('/content/drive')`
  lalu set `X_PATH`/`Y_PATH` ke path di Drive.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# X_PATH = '/content/drive/MyDrive/bisindo/X.npy'   # contoh, sesuaikan
# Y_PATH = '/content/drive/MyDrive/bisindo/y.npy'

X = np.load(X_PATH)
y = np.load(Y_PATH)
print("X:", X.shape, "| y:", y.shape)

# sanity check dimensi sesuai pipeline preprocessing
assert X.ndim == 3 and X.shape[1] == SEQUENCE_LENGTH and X.shape[2] == N_FEATURES, \
    f"Shape X harus (n,{SEQUENCE_LENGTH},{N_FEATURES}), dapat {X.shape}"

# distribusi sample per kelas (cek keseimbangan -> stratify penting)
uniq, cnt = np.unique(y, return_counts=True)
plt.figure(figsize=(12, 4))
plt.bar(uniq, cnt)
plt.xlabel("indeks kelas"); plt.ylabel("jumlah sample")
plt.title("Distribusi sample per kelas")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png")); plt.show()

## 4. Label map (indeks → kata)
Untuk classification report & confusion matrix yang terbaca.

In [ ]:
def load_labels(path, num_classes):
    """Baca label_map.json {index: kata}; fallback ke LABEL_xx bila gagal."""
    try:
        with open(path, encoding="utf-8") as f:
            m = json.load(f)
        return [m[str(i)] for i in range(num_classes)]
    except Exception as e:
        print("label_map tak terbaca, fallback LABEL_xx:", e)
        return [f"LABEL_{i:02d}" for i in range(num_classes)]

In [ ]:
LABELS = load_labels(LABEL_MAP_PATH, NUM_CLASSES)
print("Contoh label:", LABELS[:5])

## 5. Split stratified 70/15/15
Stratifikasi menjaga proporsi tiap kelas seimbang di train/val/test — krusial
untuk dataset kecil. Test di-*hold out* dan TIDAK dipakai saat training/tuning.

In [ ]:
def make_splits(X, y, val_size=0.15, test_size=0.15, seed=42):
    """Kembalikan train/val/test (stratified). val_size & test_size relatif total."""
    # 1) sisihkan test
    X_tmp, X_test, y_tmp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed)
    # 2) sisihkan val dari sisa (skala ulang proporsi)
    val_rel = val_size / (1.0 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tmp, y_tmp, test_size=val_rel, stratify=y_tmp, random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(
    X, y, val_size=0.15, test_size=0.15, seed=SEED)
print(f"train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")

# one-hot untuk categorical_crossentropy
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

## 6. (Opsional) Augmentasi landmark
Aktif hanya bila `AUGMENT = True`. Teknik (semua pada koordinat, murah):
- **jitter** — noise Gaussian kecil tiap titik (robust ke noise deteksi)
- **scaling** — skala global acak ±10% (variasi jarak ke kamera)
- **time-shift** — geser temporal (variasi timing gestur)
- **mirror + swap tangan** — cermin sumbu-x & tukar blok tangan kiri↔kanan
  (signer kidal/tidak; gandakan variasi)

In [ ]:
def augment_sequence(seq, noise=0.01, scale_range=0.1, max_shift=3):
    """Augmentasi satu sequence (SEQUENCE_LENGTH, 126) -> shape sama."""
    out = seq.astype(np.float32).copy()
    out = out + np.random.normal(0.0, noise, out.shape).astype(np.float32)  # jitter
    out = out * (1.0 + np.random.uniform(-scale_range, scale_range))        # scaling
    shift = np.random.randint(-max_shift, max_shift + 1)                    # time-shift
    if shift != 0:
        out = np.roll(out, shift, axis=0)
    return out.astype(np.float32)

def mirror_swap(seq):
    """Cermin sumbu-x + tukar tangan kiri<->kanan. seq (T,126)."""
    s = seq.reshape(seq.shape[0], 2, 21, 3).copy()  # [kiri|kanan] x 21 x (x,y,z)
    s[..., 0] *= -1.0          # mirror x
    s = s[:, ::-1, :, :]       # tukar blok tangan
    return s.reshape(seq.shape[0], -1).astype(np.float32)

def augment_training_set(X, y, factor=1, mirror=True):
    """Perbanyak data latih. factor = jumlah salinan ter-augment per sampel."""
    Xs, ys = [X], [y]
    for _ in range(factor):
        Xs.append(np.stack([augment_sequence(s) for s in X])); ys.append(y)
    if mirror:
        Xs.append(np.stack([mirror_swap(s) for s in X])); ys.append(y)
    return np.concatenate(Xs).astype(np.float32), np.concatenate(ys)

In [ ]:
if AUGMENT:
    n_before = len(X_train)
    X_train, y_train = augment_training_set(X_train, y_train, factor=1, mirror=True)
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    print(f"Augmentasi: {n_before} -> {len(X_train)} sample latih")
else:
    print("Augmentasi OFF (baseline). Set AUGMENT=True jika val accuracy rendah.")

## 7. Arsitektur LSTM
Sengaja **sederhana** (anti over-engineering): 2 LSTM bertingkat + Dropout +
Dense softmax 40 kelas.

> Catatan: `recurrent_dropout` menonaktifkan kernel cuDNN (training sedikit lebih
> lambat di GPU) tetapi memperkuat regularisasi. Set ke 0.0 bila ingin lebih cepat.

In [ ]:
def build_model(seq_len=SEQUENCE_LENGTH, n_features=N_FEATURES,
                n_classes=NUM_CLASSES, lr=LEARNING_RATE):
    model = models.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        layers.Dropout(0.4),
        layers.LSTM(64),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
model = build_model()
model.summary()

## 8. Callbacks & training
- **EarlyStopping** — stop bila val_loss tak membaik 15 epoch, kembalikan bobot terbaik
- **ModelCheckpoint** — simpan model val_accuracy terbaik
- **ReduceLROnPlateau** — turunkan LR saat val_loss stagnan

In [ ]:
def make_callbacks(output_dir, patience=15):
    ckpt = os.path.join(output_dir, "bisindo_lstm_best.keras")
    return [
        EarlyStopping(monitor='val_loss', patience=patience,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(ckpt, monitor='val_accuracy',
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=6, min_lr=1e-6, verbose=1),
    ]

In [ ]:
callbacks = make_callbacks(OUTPUT_DIR)
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1,
)

## 9. Grafik accuracy & loss
Gap train vs val = indikator overfitting (untuk laporan).

In [ ]:
def plot_history(history, output_dir):
    h = history.history
    for metric, fname, title in [
        ('accuracy', 'training_accuracy.png', 'Accuracy per epoch'),
        ('loss',     'training_loss.png',     'Loss per epoch'),
    ]:
        plt.figure(figsize=(8, 5))
        plt.plot(h[metric], label='train')
        plt.plot(h['val_' + metric], label='val')
        plt.title(title); plt.xlabel('epoch'); plt.ylabel(metric)
        plt.legend(); plt.grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(os.path.join(output_dir, fname)); plt.show()

In [ ]:
plot_history(history, OUTPUT_DIR)

## 10. Evaluasi test: classification report + confusion matrix

In [ ]:
def evaluate_and_report(model, X_test, y_test_int, labels, output_dir):
    """Evaluasi di test set; simpan report + confusion matrix; balikan metrik."""
    y_cat = to_categorical(y_test_int, num_classes=len(labels))
    loss, acc = model.evaluate(X_test, y_cat, verbose=0)
    print(f"Test loss: {loss:.4f} | Test accuracy: {acc:.4f}\n")

    y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    txt = classification_report(y_test_int, y_pred, target_names=labels, zero_division=0)
    print(txt)
    with open(os.path.join(output_dir, 'classification_report.txt'), 'w', encoding='utf-8') as f:
        f.write(txt)
    rep = classification_report(y_test_int, y_pred, target_names=labels,
                                zero_division=0, output_dict=True)

    cm = confusion_matrix(y_test_int, y_pred, labels=list(range(len(labels))))
    plt.figure(figsize=(14, 12))
    if _HAS_SNS:
        sns.heatmap(cm, annot=False, cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
    else:
        plt.imshow(cm, cmap='Blues'); plt.colorbar()
    plt.title('Confusion Matrix (test set)')
    plt.xlabel('prediksi'); plt.ylabel('aktual')
    plt.tight_layout(); plt.savefig(os.path.join(output_dir, 'confusion_matrix.png')); plt.show()

    return {'test_loss': float(loss), 'test_accuracy': float(acc),
            'macro_f1': float(rep['macro avg']['f1-score']),
            'weighted_f1': float(rep['weighted avg']['f1-score'])}

In [ ]:
metrics = evaluate_and_report(model, X_test, y_test, LABELS, OUTPUT_DIR)
metrics

## 11. Export model & ringkasan metrik
Simpan `.keras` (native Keras 3) **dan** `.h5` (dipakai inference
`predictor.py` via `config.MODEL_PATH`).

In [ ]:
def export_artifacts(model, metrics, labels, history, output_dir):
    keras_path = os.path.join(output_dir, 'bisindo_lstm.keras')
    h5_path    = os.path.join(output_dir, 'bisindo_lstm.h5')
    model.save(keras_path)
    try:
        model.save(h5_path)            # legacy HDF5 (Keras 3 masih dukung)
    except Exception as e:
        print("Peringatan simpan .h5:", e)

    best_epoch = (int(np.argmax(history.history['val_accuracy'])) + 1
                  if history is not None else None)
    summary = dict(metrics)
    summary['best_epoch']  = best_epoch
    summary['num_classes'] = len(labels)
    with open(os.path.join(output_dir, 'metrics_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    with open(os.path.join(output_dir, 'label_map.json'), 'w', encoding='utf-8') as f:
        json.dump({i: l for i, l in enumerate(labels)}, f, ensure_ascii=False, indent=2)

    print("Artefak tersimpan di", output_dir, "->", sorted(os.listdir(output_dir)))

In [ ]:
export_artifacts(model, metrics, LABELS, history, OUTPUT_DIR)

## 12. Langkah berikutnya
1. **Unduh artefak** dari folder `outputs/` (atau simpan ke Drive).
2. Pindahkan ke proyek lokal `models/`:
   - `bisindo_lstm.h5`  → `models/bisindo_lstm.h5`
   - `label_map.json`   → `models/label_map.json`
3. Jalankan inference lokal: `python app.py`.

**Jika akurasi validation/test rendah:**
- set `AUGMENT = True` lalu latih ulang
- kecilkan unit LSTM (128→64, 64→32) bila gap train-val besar (overfitting)
- tambah data / signer berbeda untuk generalisasi
- naikkan `patience` EarlyStopping bila kurva masih menanjak saat berhenti